# Notebook 2 — LIAR | 5 representações

CV, TF-IDF, Word2Vec, FastText e MiniLM. Preserva os splits originais. Os quatro primeiros são ajustados no treino; MiniLM é pré-treinado. Projeções 2D coloridas.

## Instalação

In [ ]:
%pip -q install numpy pandas requests nltk gensim scipy scikit-learn sentence-transformers pyarrow joblib matplotlib

## Configuração e classes

In [ ]:
import io, re, json, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import nltk
from nltk.corpus import stopwords
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec, FastText
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import joblib
SAVE_MODELS = False  # True apenas se precisar gerar embeddings de novos textos
SEED=42
np.random.seed(SEED)
LIAR_URL='https://www.cs.ucsb.edu/~william/data/liar_dataset.zip'
ROOT=Path('/content') if Path('/content').exists() else Path('.')
OUTPUT_DIR=ROOT/'data'/'embeddings'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
SPLITS=('train','validation','test')
DATASETS=('liar',)
EMBEDDINGS=('cv','tfidf','w2v','fasttext','bert')
LABEL_MAPS={
 'liar':{'pants-fire':0,'false':1,'barely-true':2,'half-true':3,'mostly-true':4,'true':5}}
CLASS_GRAY_COLORS={'liar':{0:'#252525',1:'#525252',2:'#737373',3:'#969696',4:'#636363',5:'#404040'}}
nltk.download('stopwords',quiet=True)
STOPWORDS=set(stopwords.words('english'))-{'no','not','nor','never'}
NEGATIONS={'no','not','nor','never'}

## Carregamento dos arquivos originais

In [ ]:
def load_liar():
    response = requests.get(LIAR_URL, timeout=120)
    response.raise_for_status()
    filenames = {"train": "train.tsv", "validation": "valid.tsv", "test": "test.tsv"}
    splits = {}
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        for split, filename in filenames.items():
            member = next((name for name in archive.namelist() if name.endswith(filename)), None)
            if member is None:
                raise FileNotFoundError(f"Arquivo {filename} ausente no ZIP")
            with archive.open(member) as f:
                df = pd.read_csv(f, sep="\t", header=None, dtype=str)
            df = df.iloc[:, [2, 1]].copy()
            df.columns = ["statement", "original_label"]
            df["statement"] = df["statement"].fillna("").astype(str)
            df["original_label"] = df["original_label"].str.strip().str.lower()
            for name, mapping in LABEL_MAPS.items():
                df[name] = df["original_label"].map(mapping)
                if df[name].isna().any():
                    raise ValueError(f"Rótulos não reconhecidos no split {split}: {name}")
                df[name] = df[name].astype(int)
            splits[split] = df.reset_index(drop=True)
    return splits

## Limpeza textual

In [ ]:
def clean_text(text):
    """Limpeza do EDA, com preservação explícita das negações."""
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(
        word for word in text.split()
        if (len(word) > 2 or word in NEGATIONS) and word not in STOPWORDS
    )

## Extração das cinco representações

In [ ]:
def average_w2v(model,texts):
    dim=model.vector_size
    result=np.zeros((len(texts),dim),dtype=np.float32)
    for i,t in enumerate(texts):
        vec=[model.wv[w] for w in t.split() if w in model.wv]
        if vec: result[i]=np.mean(vec,axis=0)
    return result

def build_embeddings():

    # ========================================================
    # 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
    # ========================================================

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    splits = load_liar()

    for split, df in splits.items():
        df["clean_text"] = df["statement"].fillna("").map(clean_text)

        df.to_parquet(
            OUTPUT_DIR / f"{split}_texts_labels.parquet",
            index=False
        )

        print(f"{split}: {len(df)} registros")

    train_text = splits["train"]["clean_text"].tolist()

    # ========================================================
    # 2. COUNT VECTORIZER E TF-IDF
    # ========================================================

    vectorizers = {
        "cv": CountVectorizer(
            max_features=20000,
            token_pattern=r"(?u)\b\w+\b"
        ),

        "tfidf": TfidfVectorizer(
            max_features=20000,
            sublinear_tf=True,
            token_pattern=r"(?u)\b\w+\b"
        )
    }

    for key, vectorizer in vectorizers.items():

        print(f"\nGerando {key.upper()}...")

        # Ajustar apenas no treinamento
        vectorizer.fit(train_text)

        if SAVE_MODELS:
            joblib.dump(
                vectorizer,
                OUTPUT_DIR / f"{key}_vectorizer.joblib"
            )

        for split, df in splits.items():

            X = vectorizer.transform(df["clean_text"])

            sparse.save_npz(
                OUTPUT_DIR / f"{split}_{key}.npz",
                X
            )

            print(f"{key.upper()} | {split}: {X.shape}")

    # ========================================================
    # 3. WORD2VEC
    # ========================================================
    print("\nTreinando Word2Vec...")
    train_tokens = [
        text.split()
        for text in train_text
    ]
    w2v = Word2Vec(
        sentences=train_tokens,
        vector_size=100,
        window=5,
        min_count=1,
        workers=1,
        sg=1,
        epochs=10,
        seed=SEED
    )
    if SAVE_MODELS:
        w2v.save(
            str(OUTPUT_DIR / "word2vec.model")
        )
    for split, df in splits.items():

        X_w2v = average_w2v(
            w2v,
            df["clean_text"].tolist()
        )

        np.save(
            OUTPUT_DIR / f"{split}_w2v.npy",
            X_w2v
        )
        print(f"W2V | {split}: {X_w2v.shape}")
    # ========================================================
    # 4. FASTTEXT
    # ========================================================
    print("\nTreinando FastText...")
    fasttext = FastText(
        sentences=train_tokens,
        vector_size=100,
        window=5,
        min_count=1,
        workers=1,
        sg=1,
        epochs=10,
        seed=SEED
    )
    if SAVE_MODELS:
        fasttext.save(
            str(OUTPUT_DIR / "fasttext.model")
        )
    for split, df in splits.items():
        X_fasttext = average_w2v(
            fasttext,
            df["clean_text"].tolist()
        )
        np.save(
            OUTPUT_DIR / f"{split}_fasttext.npy",
            X_fasttext
        )
        print(f"FastText | {split}: {X_fasttext.shape}")
    # ========================================================
    # 5. BERT — MiniLM
    # ========================================================
    print("\nGerando embeddings BERT (MiniLM)...")
    bert = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )
    for split, df in splits.items():
        texts = df["statement"].fillna("").astype(str).tolist()
        arr = bert.encode(
            texts,
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        X_bert = arr.astype(np.float32)
        np.save(
            OUTPUT_DIR / f"{split}_bert.npy",
            X_bert
        )
        print(f"BERT | {split}: {X_bert.shape}")
    # ========================================================
    # 6. METADADOS
    # ========================================================
    meta = {
        "datasets": list(DATASETS),
        "embeddings": list(EMBEDDINGS),
        "seed": SEED,
        "bert_model": (
            "sentence-transformers/all-MiniLM-L6-v2"
        ),
        "w2v_dim": 100,
        "fasttext_dim": 100,
        "note": (
            "CV e TF-IDF ajustados somente no treino; "
            "Word2Vec e FastText treinados somente no treino; "
            "MiniLM pré-treinado."
        )
    }
    (OUTPUT_DIR / "metadata.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("\n" + "=" * 60)
    print("Embeddings gerados com sucesso!")
    print("Diretório:", OUTPUT_DIR)
    print("Datasets:", DATASETS)
    print("Representações:", EMBEDDINGS)

In [ ]:
build_embeddings()

## Projeções 2D

In [ ]:

from sklearn.decomposition import TruncatedSVD, PCA
from matplotlib.lines import Line2D
from scipy import sparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

EMBEDDINGS = ('cv', 'tfidf', 'w2v', 'fasttext', 'bert')

# Nomes das classes
# Ajuste a ordem do LIAR se sua função load_liar()
# utilizar outro mapeamento numérico.
CLASS_NAMES = {
    'liar': {
        0: 'pants-fire',
        1: 'false',
        2: 'barely-true',
        3: 'half-true',
        4: 'mostly-true',
        5: 'true'
    }
}

# Cores diferentes e visualmente distinguíveis
CLASS_COLORS = {
    'liar': {
        0: '#D62728',  # Vermelho
        1: '#FF7F0E',  # Laranja
        2: '#9467BD',  # Roxo
        3: '#1F77B4',  # Azul
        4: '#2CA02C',  # Verde
        5: '#8C564B'   # Marrom
    }
}

# Marcadores diferentes
CLASS_MARKERS = {
    0: 'o',  # Círculo
    1: 'X',  # X preenchido
    2: 's',  # Quadrado
    3: '^',  # Triângulo
    4: 'D',  # Losango
    5: '*'   # Estrela
}

# ============================================================
# 2. CARREGAR OS RÓTULOS E SELECIONAR A AMOSTRA
# ============================================================

df_train = pd.read_parquet(
    OUTPUT_DIR / 'train_texts_labels.parquet'
)

rng = np.random.default_rng(SEED)

ix = rng.choice(
    len(df_train),
    size=min(1500, len(df_train)),
    replace=False
)

# ============================================================
# 3. CALCULAR PROJEÇÕES DOS CINCO EMBEDDINGS
# ============================================================

projections = {}

for emb in EMBEDDINGS:

    if emb in ('cv', 'tfidf'):
        X = sparse.load_npz(
            OUTPUT_DIR / f'train_{emb}.npz'
        )
    else:
        X = np.load(
            OUTPUT_DIR / f'train_{emb}.npy'
        )

    X_sample = X[ix]

    if sparse.issparse(X_sample):
        reducer = TruncatedSVD(
            n_components=2,
            random_state=SEED
        )
    else:
        reducer = PCA(
            n_components=2,
            random_state=SEED
        )

    projections[emb] = reducer.fit_transform(X_sample)

    print(f'{emb.upper()}: projeção calculada')

# ============================================================
# 4. GERAR GRÁFICOS PARA O LIAR
# ============================================================

for dataset in ('liar',):

    y = df_train[dataset].to_numpy()[ix]
    classes = sorted(np.unique(y))

    fig, axes = plt.subplots(
        1, 5,
        figsize=(23, 5.5)
    )

    for ax, emb in zip(axes, EMBEDDINGS):

        projected = projections[emb]

        for cl in classes:

            pts = projected[y == cl]

            ax.scatter(
                pts[:, 0],
                pts[:, 1],
                s=22,
                alpha=0.65,
                color=CLASS_COLORS[dataset][cl],
                marker=CLASS_MARKERS[cl],
                edgecolors='none',
                label=CLASS_NAMES[dataset][cl]
            )

        ax.set_title(
            emb.upper(),
            fontsize=12,
            fontweight='bold'
        )

        ax.set_xlabel('Componente 1')
        ax.set_ylabel('Componente 2')
        ax.grid(alpha=0.15)

    # ========================================================
    # 5. LEGENDA ÚNICA, COM NOME E MARCADOR DE CADA CLASSE
    # ========================================================

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker=CLASS_MARKERS[cl],
            linestyle='None',
            markerfacecolor=CLASS_COLORS[dataset][cl],
            markeredgecolor=CLASS_COLORS[dataset][cl],
            markersize=9,
            label=CLASS_NAMES[dataset][cl]
        )
        for cl in classes
    ]

    fig.legend(
        handles=legend_handles,
        title='Classes',
        loc='lower center',
        bbox_to_anchor=(0.5, 0.015),
        ncol=len(classes),
        fontsize=10,
        title_fontsize=11,
        frameon=False
    )

    dataset_title = 'LIAR'

    fig.suptitle(
        f'Representações textuais — {dataset_title}',
        fontsize=16,
        fontweight='bold',
        y=0.98
    )

    # Reservar espaço para a legenda
    fig.subplots_adjust(
        left=0.04,
        right=0.99,
        top=0.84,
        bottom=0.24,
        wspace=0.28
    )

    filename = (
        f'representacoes_5_embeddings_{dataset}_colorido.jpg'
    )

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )

    plt.show()
    plt.close(fig)

    print(f'Imagem salva: {OUTPUT_DIR / filename}')


## Exportar ZIP para notebooks 3 e 4

Exporta somente as matrizes dos cinco embeddings, textos, rótulos e metadados. Modelos e gráficos ficam fora do ZIP. A estrutura interna `embeddings/` é preservada. Se os arquivos já estiverem disponíveis na sessão, basta executar esta célula.

In [ ]:
import zipfile
from pathlib import Path

ROOT = Path('/content') if Path('/content').exists() else Path('.')
OUTPUT_DIR = ROOT / 'data' / 'embeddings'
ZIP_PATH = ROOT / 'embeddings_liar.zip'
TEMP_PATH = ZIP_PATH.with_suffix('.zip.tmp')
SPLITS = ('train', 'validation', 'test')
EMBEDDINGS = ('cv', 'tfidf', 'w2v', 'fasttext', 'bert')

# Selecionar somente os arquivos necessários
arquivos = [OUTPUT_DIR / 'metadata.json']
for split in SPLITS:
    arquivos.append(OUTPUT_DIR / f'{split}_texts_labels.parquet')
    for emb in EMBEDDINGS:
        ext = 'npz' if emb in ('cv', 'tfidf') else 'npy'
        arquivos.append(OUTPUT_DIR / f'{split}_{emb}.{ext}')

faltantes = [str(p) for p in arquivos if not p.is_file() or p.stat().st_size == 0]
if faltantes:
    raise FileNotFoundError('Arquivos ausentes ou vazios:\n' + '\n'.join(faltantes))

# Criar arquivo temporário sem compressão adicional
try:
    with zipfile.ZipFile(TEMP_PATH, 'w', compression=zipfile.ZIP_STORED) as z:
        for arquivo in arquivos:
            z.write(arquivo, arcname=f'embeddings/{arquivo.name}')

    # Validar nomes e CRC antes de substituir o ZIP anterior
    with zipfile.ZipFile(TEMP_PATH, 'r') as z:
        esperados = {f'embeddings/{p.name}' for p in arquivos}
        if set(z.namelist()) != esperados:
            raise RuntimeError('Conteúdo inesperado no ZIP.')
        bad_file = z.testzip()
        if bad_file is not None:
            raise RuntimeError(f'Arquivo corrompido: {bad_file}')

    TEMP_PATH.replace(ZIP_PATH)
finally:
    TEMP_PATH.unlink(missing_ok=True)

print('ZIP gerado e validado com sucesso.')
print(f'Arquivos: {len(arquivos)}')
print(f'Tamanho: {ZIP_PATH.stat().st_size / 1024**2:.2f} MiB')
print(f'Caminho: {ZIP_PATH.resolve()}')


In [ ]:
# Download separado: execute novamente apenas esta célula se a transferência falhar.
from google.colab import files
files.download(str(ZIP_PATH))